In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e1/sample_submission.csv
/kaggle/input/playground-series-s5e1/train.csv
/kaggle/input/playground-series-s5e1/test.csv


In [2]:
!pip install scikit-learn==1.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 74.8 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2


In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline

In [3]:
from tqdm.rich import tqdm

tqdm.pandas()

In [4]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e1/train.csv')
df_train

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [ ]:
df_test = pd.read_csv('/kaggle/input/playground-series-s5e1/test.csv')
ids = df_test['id']

In [ ]:
# df = pd.concat([df_train, df_test])
# df

In [ ]:
# df_test

In [ ]:
# df_train = df

In [ ]:
print(df_train.dtypes)

In [5]:
df_train.drop('id', axis=1, inplace=True)

In [ ]:
0.8 * 328680

In [ ]:
df_train.nunique()

In [6]:
# df_train['year'] = df_train['date'].apply(lambda x: int(x[:4]))
df_train['month'] = df_train['date'].apply(lambda x: int(x[5:7]))
df_train['day'] = df_train['date'].apply(lambda x: int(x[8:10]))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['day of year'] = df_train['date'].dt.dayofyear
df_train['day of week'] = df_train['date'].dt.weekday
df_train['sin day of week'] = np.sin(2 * np.pi * df_train['day of week'] / 7)
df_train['cos day of week'] = np.cos(2 * np.pi * df_train['day of week'] / 7)
#df_train['seconds since 1970'] = df_train['date'].astype('int64') // 10 ** 9
df_train.drop('date', axis=1, inplace=True) #
df_train

,country,store,product,num_sold,month,day,day of year,day of week,sin day of week,cos day of week
0,Canada,Discount Stickers,Holographic Goose,NaN,1,1,1,4,-0.433884,-0.900969
1,Canada,Discount Stickers,Kaggle,973.0,1,1,1,4,-0.433884,-0.900969
2,Canada,Discount Stickers,Kaggle Tiers,906.0,1,1,1,4,-0.433884,-0.900969
3,Canada,Discount Stickers,Kerneler,423.0,1,1,1,4,-0.433884,-0.900969
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,1,1,1,4,-0.433884,-0.900969
...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,12,31,366,5,-0.974928,-0.222521
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,12,31,366,5,-0.974928,-0.222521
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,12,31,366,5,-0.974928,-0.222521
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,12,31,366,5,-0.974928,-0.222521


In [43]:
def season(month):
    match month:
        case 1 | 12: 
            return 'winter'
        case 2 | 3 | 4 | 5:
            return 'spring'
        case 6 | 7 | 8 :
            return 'summer'
        case _:
            return 'autumn'

df_train['season'] = df_train['month'].apply(season) 

In [34]:
df_train

,country,store,product,num_sold,month,day,day of year,day of week,sin day of week,cos day of week,country-store-product-month-dayofweek,season
0,Canada,Discount Stickers,Holographic Goose,NaN,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersHolographic Goose1-4,winter
1,Canada,Discount Stickers,Kaggle,973.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKaggle1-4,winter
2,Canada,Discount Stickers,Kaggle Tiers,906.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKaggle Tiers1-4,winter
3,Canada,Discount Stickers,Kerneler,423.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKerneler1-4,winter
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKerneler Dark Mode1-4,winter
...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartHolographic Goose...,winter
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKaggle12-5,winter
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKaggle Tiers12-5,winter
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKerneler12-5,winter


In [50]:
df_train['country-store-product-season-dayofweek'] = df_train['country'].apply(lambda a: a + '-') \
    + df_train['store'].apply(lambda a: a + '-') \
    + df_train['product'].apply(lambda a: a + '-') \
    + df_train['season'].apply(lambda a: a + '-') \
    + df_train['day of week'].apply(lambda a: str(a))  

df_train

,country,store,product,num_sold,month,day,day of year,day of week,sin day of week,cos day of week,country-store-product-month-dayofweek,season,country-store-product-season-dayofweek
0,Canada,Discount Stickers,Holographic Goose,NaN,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersHolographic Goose1-4,winter,Canada-Discount Stickers-Holographic Goose-win...
1,Canada,Discount Stickers,Kaggle,973.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKaggle1-4,winter,Canada-Discount Stickers-Kaggle-winter-4
2,Canada,Discount Stickers,Kaggle Tiers,906.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKaggle Tiers1-4,winter,Canada-Discount Stickers-Kaggle Tiers-winter-4
3,Canada,Discount Stickers,Kerneler,423.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKerneler1-4,winter,Canada-Discount Stickers-Kerneler-winter-4
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,1,1,1,4,-0.433884,-0.900969,CanadaDiscount StickersKerneler Dark Mode1-4,winter,Canada-Discount Stickers-Kerneler Dark Mode-wi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartHolographic Goose...,winter,Singapore-Premium Sticker Mart-Holographic Goo...
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKaggle12-5,winter,Singapore-Premium Sticker Mart-Kaggle-winter-5
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKaggle Tiers12-5,winter,Singapore-Premium Sticker Mart-Kaggle Tiers-wi...
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,12,31,366,5,-0.974928,-0.222521,SingaporePremium Sticker MartKerneler12-5,winter,Singapore-Premium Sticker Mart-Kerneler-winter-5


In [51]:
agg_df = df_train.groupby('country-store-product-season-dayofweek').agg(
    pred = ('num_sold', lambda x: x.sum() / x.count())
)
agg_df

<ipython-input-51-c978476aabac>:2: RuntimeWarning: invalid value encountered in scalar divide
  pred = ('num_sold', lambda x: x.sum() / x.count())


,pred
country-store-product-season-dayofweek,
Canada-Discount Stickers-Holographic Goose-autumn-0,NaN
Canada-Discount Stickers-Holographic Goose-autumn-1,NaN
Canada-Discount Stickers-Holographic Goose-autumn-2,NaN
Canada-Discount Stickers-Holographic Goose-autumn-3,NaN
Canada-Discount Stickers-Holographic Goose-autumn-4,NaN
...,...
Singapore-Stickers for Less-Kerneler-winter-2,719.442623
Singapore-Stickers for Less-Kerneler-winter-3,715.301587
Singapore-Stickers for Less-Kerneler-winter-4,751.269841


In [52]:
agg_df.count()

pred    2464
dtype: int64

In [54]:
from sklearn.impute import SimpleImputer, KNNImputer

imputer = SimpleImputer(strategy='mean')
pred = imputer.fit_transform(agg_df)
agg_df['pred'] = pred
agg_df

,pred
country-store-product-season-dayofweek,
Canada-Discount Stickers-Holographic Goose-autumn-0,742.920556
Canada-Discount Stickers-Holographic Goose-autumn-1,742.920556
Canada-Discount Stickers-Holographic Goose-autumn-2,742.920556
Canada-Discount Stickers-Holographic Goose-autumn-3,742.920556
Canada-Discount Stickers-Holographic Goose-autumn-4,742.920556
...,...
Singapore-Stickers for Less-Kerneler-winter-2,719.442623
Singapore-Stickers for Less-Kerneler-winter-3,715.301587
Singapore-Stickers for Less-Kerneler-winter-4,751.269841


In [55]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e1/test.csv')

In [56]:
df_train['year'] = df_train['date'].apply(lambda x: int(x[:4]))
df_train['month'] = df_train['date'].apply(lambda x: int(x[5:7]))
df_train['day'] = df_train['date'].apply(lambda x: int(x[8:10]))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['day of year'] = df_train['date'].dt.dayofyear
df_train['day of week'] = df_train['date'].dt.weekday
df_train['sin day of week'] = np.sin(2 * np.pi * df_train['day of week'] / 7)
df_train['cos day of week'] = np.cos(2 * np.pi * df_train['day of week'] / 7)
#df_train['seconds since 1970'] = df_train['date'].astype('int64') // 10 ** 9
df_train.drop('date', axis=1, inplace=True) #
df_train

,id,country,store,product,month,day,day of year,day of week,sin day of week,cos day of week
0,230130,Canada,Discount Stickers,Holographic Goose,1,1,1,6,-0.781831,0.62349
1,230131,Canada,Discount Stickers,Kaggle,1,1,1,6,-0.781831,0.62349
2,230132,Canada,Discount Stickers,Kaggle Tiers,1,1,1,6,-0.781831,0.62349
3,230133,Canada,Discount Stickers,Kerneler,1,1,1,6,-0.781831,0.62349
4,230134,Canada,Discount Stickers,Kerneler Dark Mode,1,1,1,6,-0.781831,0.62349
...,...,...,...,...,...,...,...,...,...,...
98545,328675,Singapore,Premium Sticker Mart,Holographic Goose,12,31,365,1,0.781831,0.62349
98546,328676,Singapore,Premium Sticker Mart,Kaggle,12,31,365,1,0.781831,0.62349
98547,328677,Singapore,Premium Sticker Mart,Kaggle Tiers,12,31,365,1,0.781831,0.62349
98548,328678,Singapore,Premium Sticker Mart,Kerneler,12,31,365,1,0.781831,0.62349


In [58]:
df_train['season'] = df_train['month'].apply(season) 

In [59]:
df_train['country-store-product-season-dayofweek'] = df_train['country'].apply(lambda a: a + '-') \
    + df_train['store'].apply(lambda a: a + '-') \
    + df_train['product'].apply(lambda a: a + '-') \
    + df_train['season'].apply(lambda a: a + '-') \
    + df_train['day of week'].apply(lambda a: str(a))  

df_train

,id,country,store,product,month,day,day of year,day of week,sin day of week,cos day of week,season,country-store-product-season-dayofweek
0,230130,Canada,Discount Stickers,Holographic Goose,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Holographic Goose-win...
1,230131,Canada,Discount Stickers,Kaggle,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kaggle-winter-6
2,230132,Canada,Discount Stickers,Kaggle Tiers,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kaggle Tiers-winter-6
3,230133,Canada,Discount Stickers,Kerneler,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kerneler-winter-6
4,230134,Canada,Discount Stickers,Kerneler Dark Mode,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kerneler Dark Mode-wi...
...,...,...,...,...,...,...,...,...,...,...,...,...
98545,328675,Singapore,Premium Sticker Mart,Holographic Goose,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Holographic Goo...
98546,328676,Singapore,Premium Sticker Mart,Kaggle,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kaggle-winter-1
98547,328677,Singapore,Premium Sticker Mart,Kaggle Tiers,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kaggle Tiers-wi...
98548,328678,Singapore,Premium Sticker Mart,Kerneler,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kerneler-winter-1


In [60]:
df_pred = pd.merge(df_train, agg_df, on='country-store-product-season-dayofweek', how='left')
df_pred

,id,country,store,product,month,day,day of year,day of week,sin day of week,cos day of week,season,country-store-product-season-dayofweek,pred
0,230130,Canada,Discount Stickers,Holographic Goose,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Holographic Goose-win...,742.920556
1,230131,Canada,Discount Stickers,Kaggle,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kaggle-winter-6,870.467742
2,230132,Canada,Discount Stickers,Kaggle Tiers,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kaggle Tiers-winter-6,718.370968
3,230133,Canada,Discount Stickers,Kerneler,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kerneler-winter-6,387.467742
4,230134,Canada,Discount Stickers,Kerneler Dark Mode,1,1,1,6,-0.781831,0.62349,winter,Canada-Discount Stickers-Kerneler Dark Mode-wi...,454.693548
...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,328675,Singapore,Premium Sticker Mart,Holographic Goose,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Holographic Goo...,291.868852
98546,328676,Singapore,Premium Sticker Mart,Kaggle,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kaggle-winter-1,1866.344262
98547,328677,Singapore,Premium Sticker Mart,Kaggle Tiers,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kaggle Tiers-wi...,1551.524590
98548,328678,Singapore,Premium Sticker Mart,Kerneler,12,31,365,1,0.781831,0.62349,winter,Singapore-Premium Sticker Mart-Kerneler-winter-1,837.377049


In [61]:
df_pred['num_sold'] = df_pred['pred']
df_pred = df_pred[['id', 'num_sold']]
df_pred.to_csv('submission.csv', index=False)
df_pred

,id,num_sold
0,230130,742.920556
1,230131,870.467742
2,230132,718.370968
3,230133,387.467742
4,230134,454.693548
...,...,...
98545,328675,291.868852
98546,328676,1866.344262
98547,328677,1551.524590
98548,328678,837.377049


In [62]:
pred

array([[742.92055604],
       [742.92055604],
       [742.92055604],
       ...,
       [751.26984127],
       [799.6984127 ],
       [874.5483871 ]])

In [ ]:
X = pd.read_csv('/kaggle/input/playground-series-s5e1/train.csv')

In [ ]:
from catboost import CatBoostRegressor, Pool



In [41]:
agg_df_1 = df_train.groupby('country-store-product-month-dayofweek').agg(
    pred = ('num_sold', lambda x: x.sum() / x.count())
)
agg_df_1

<ipython-input-41-5be75b7ef54c>:2: RuntimeWarning: invalid value encountered in scalar divide
  pred = ('num_sold', lambda x: x.sum() / x.count())


,pred
country-store-product-month-dayofweek,
CanadaDiscount StickersHolographic Goose1-0,NaN
CanadaDiscount StickersHolographic Goose1-1,NaN
CanadaDiscount StickersHolographic Goose1-2,NaN
CanadaDiscount StickersHolographic Goose1-3,NaN
CanadaDiscount StickersHolographic Goose1-4,NaN
...,...
SingaporeStickers for LessKerneler9-2,797.800000
SingaporeStickers for LessKerneler9-3,805.806452
SingaporeStickers for LessKerneler9-4,855.166667


In [42]:
agg_df_1.count()

pred    7384
dtype: int64

In [ ]:
X = df_train

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class DropNaNColumns(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        # Calculate the percentage of NaN values in each column
        nan_percentage = X.isna().mean()
        # Identify columns to drop
        self.columns_to_drop_ = nan_percentage[nan_percentage > self.threshold].index
        return self

    def transform(self, X):
        # Drop the identified columns
        return X.drop(columns=self.columns_to_drop_)


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def preprocess_data(X, threshold=0.5, n_neighbors=1500, encode=True, impute=True):
    # Separate numerical and categorical columns
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns


    feature_names = X.columns
    # Numerical feature pipeline: Imputation + Scaling
    if impute:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            ('scaler', StandardScaler())  # Scale the numerical features
        ])
    else:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            #('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            ('scaler', StandardScaler())  # Scale the numerical features
        ])


    # Categorical feature pipeline: Imputation + Encoding
    if encode:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            # ('imputer', SimpleImputer(strategy='constant', fill_value='.')),#SimpleImputer(strategy='constant', fill_value='.')), 
            ('encoder', OneHotEncoder(handle_unknown='ignore')) 
        ])
    else:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', SimpleImputer(strategy='constant', fill_value='.')), 
        ])

    # Define transformers
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline, numerical_features),
            ('cat', categorical_pipeline, categorical_features)
        ],
        verbose=True,
        n_jobs=-1
    )

    return preprocessor

In [ ]:
preprocessor = preprocess_data(X, threshold=0.8, encode=True, impute=False)

In [ ]:
preprocessor.fit(X)

In [ ]:
num_features = preprocessor.output_indices_['num']
cat_features = preprocessor.output_indices_['cat']
num_features, cat_features

In [ ]:
df = pd.DataFrame(preprocessor.transform(X))
df

In [ ]:
preprocessor = preprocess_data(X, threshold=0.8, encode=True, impute=True)

In [ ]:
preprocessor.fit(X)

In [ ]:
df = pd.DataFrame(preprocessor.transform(X))
df

In [ ]:
df.iloc[230130:, 0].astype('float')

In [ ]:
df_pred = pd.DataFrame(df.iloc[230130:, 0].astype('float'), columns=['num_sold'])
df_pred.insert(0, 'id', ids)
df_pred.to_csv('submission.csv', index=False)
df_pred

In [ ]:
# df_train.drop('num_sold', axis=1, inplace=True)
# df_train

In [ ]:
# target = df.iloc[:, 0].astype('float')
# target

In [ ]:
# target.count(), type(target[0]), target[0]

In [ ]:
# X, y = df_train, target

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# regressor_1 = SGDRegressor(random_state=42)
# regressor_2 = MLPRegressor(max_iter=100, random_state=42)
# regressor_3 = RandomForestRegressor(random_state=42)
regressor_1 = XGBRegressor(n_estimators=500, device='cuda', random_state=42, n_jobs=-1)
regressor_2 = LGBMRegressor(n_estimators=500, random_state=42, verbose=-1, n_jobs=-1, device='gpu') #  device='gpu', gpu_device_id=0
regressor_3 = CatBoostRegressor(iterations=500, loss_function='MAPE', random_seed=42, verbose=100, thread_count=-1, task_type='GPU') # task_type='GPU' loss_function=RmsleObjective(), eval_metric='RMSE'

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class ClassifierWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y, eval_set=None):
        if eval_set is not None:
            self.estimator.fit(X, y, eval_set=eval_set)
        self.estimator.fit(X, y)
        return self

    def predict(self, X):
        return self.estimator.predict(X)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    @property
    def classes_(self):
        return self.estimator.classes_

    def __sklearn_is_fitted__(self):
        if self.estimator.classes_ is None:
            return False
        return True

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class RegressorWrapper(RegressorMixin, BaseEstimator):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y, eval_set=None, baseline=None, fobj=None):
        if baseline is not None or eval_set is not None:
          self.estimator.fit(X, y, eval_set=eval_set, baseline=baseline)
        if fobj is not None:
          self.estimator.fit(X, y, fobj=fobj)
        else:
          self.estimator.fit(X, y)
        return self

    def predict(self, X):
        return self.estimator.predict(X)


    @property
    def feature_importances_(self):
        if hasattr(self.estimator, 'feature_importances_'):
            return self.estimator.feature_importances_
        else:
            raise AttributeError("The underlying estimator does not have feature_importances_.")

    def __sklearn_is_fitted__(self):
        return hasattr(self.estimator, 'coef_') or hasattr(self.estimator, 'feature_importances_')

In [ ]:
from sklearn.base import clone

models_ = {
    'xgboost': regressor_1,
    'lgbm': regressor_2,
    'catboost': regressor_3
}

In [ ]:
from sklearn.base import clone, is_classifier, is_regressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

class SklearnPipelineBuilder:
    def __init__(self, model, preprocessor, wrapper=None):
        self.model = clone(model)
        self.preprocessor = preprocessor
        self.task = 'regressor' #if is_classifier(self.model) else 'regressor'
        self.wrapper = wrapper

    def build_pipeline(self):
        steps = []
        print(self.task)
        steps.append(('preprocessor', self.preprocessor))
        if self.wrapper:
            steps.append((self.task, self.wrapper(self.model)))
        else:
            steps.append((self.task, self.model))
        
        # Create and return the pipeline
        pipeline = Pipeline(steps)
        return pipeline


# Create a pipeline with a Random Forest model
pipeline_builder = SklearnPipelineBuilder(model=LogisticRegression(), preprocessor=preprocessor, wrapper=None)
pipeline = pipeline_builder.build_pipeline()
pipeline

In [ ]:
# models = {}
# for name, model in models_.items():
#     # if name == 'catboost_model':
#     #     preprocessor = preprocess_data(X_train, threshold=0.7, approach='catboost')
#     # else:
#     preprocessor = preprocess_data(X, threshold=0.7, encode=True, impute=False)
#     factory = SklearnPipelineBuilder(model=clone(model), preprocessor=preprocessor, wrapper=RegressorWrapper)
#     models[name] = factory.build_pipeline()

# models

In [ ]:
import optuna
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.base import clone
from sklearn.model_selection import KFold


def CV(model_name, n_folds, params):
    scores = []
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    for fold, (train_index, val_index) in enumerate(kf.split(X)):
        print(f"Fold {fold + 1} out of {n_folds}")

        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        # Create the model with trial parameters
        model = clone(models[model_name])
        model.set_params(**params)
        model.fit(X_train, y_train)

        # Evaluate using root mean squared log error
        scores += [mean_absolute_percentage_error(y_val, model.predict(X_val))]

    return np.mean(scores)

def xgboost_objective(trial):
    # Define hyperparameters to tune
    params = {
        'regressor__estimator__objective': 'reg:absoluteerror',
        # 'eval_metric': 'rmse',
        'regressor__estimator__booster': trial.suggest_categorical('regressor__estimator__booster', ['gbtree', 'dart']), # 'gbtree',
        'regressor__estimator__n_estimators': trial.suggest_int('regressor__estimator__n_estimators', 500, 1000),
        'regressor__estimator__learning_rate': trial.suggest_loguniform('regressor__estimator__learning_rate', 0.01, 0.1),
        'regressor__estimator__max_depth': trial.suggest_int('regressor__estimator__max_depth', 4, 16),
        'regressor__estimator__lambda': trial.suggest_float('regressor__estimator__lambda', 0.0001, 5, log=True),
        'regressor__estimator__alpha': trial.suggest_float('regressor__estimator__alpha', 0.0001, 5, log=True),
        'regressor__estimator__subsample': trial.suggest_float('regressor__estimator__subsample', 0.5, 1.0),
        'regressor__estimator__colsample_bytree': trial.suggest_float('regressor__estimator__colsample_bytree', 0.5, 1.0),
    }

    return CV(model_name='xgboost', n_folds=10, params=params)


def lightgbm_objective(trial):
    # Define hyperparameters to tune
    params = {
        'regressor__estimator__objective': 'mape',
        #'regressor__estimator__max_bin': trial.suggest_int('max_bin', 255, 500),
        #'metric': 'rmse',
        'regressor__estimator__boosting_type': trial.suggest_categorical('regressor__estimator__boosting_type', ['gbdt', 'dart']), # 'gbdt',
        'regressor__estimator__n_estimators': trial.suggest_int('regressor__estimator__n_estimators', 500, 1000),
        'regressor__estimator__learning_rate': trial.suggest_loguniform('regressor__estimator__learning_rate', 0.01, 0.1),
        'regressor__estimator__max_depth': trial.suggest_int('regressor__estimator__max_depth', -1, 16),
        'regressor__estimator__num_leaves': trial.suggest_int('regressor__estimator__num_leaves', 200, 512),
        'regressor__estimator__lambda_l1': trial.suggest_float('regressor__estimator__lambda_l1', 0.0001, 5, log=True),
        'regressor__estimator__lambda_l2': trial.suggest_float('regressor__estimator__lambda_l2', 0.0001, 5, log=True),
        'regressor__estimator__subsample': trial.suggest_float('regressor__estimator__subsample', 0.5, 1.0),
        'regressor__estimator__colsample_bytree': trial.suggest_float('regressor__estimator__colsample_bytree', 0.8, 1.0),
        'regressor__estimator__bagging_freq': trial.suggest_int('regressor__estimator__bagging_freq', 5, 12),
        'regressor__estimator__min_data_in_leaf': trial.suggest_int('regressor__estimator__min_data_in_leaf', 20, 100)
        # 'regressor__estimator__bagging_fraction': 0.5200288825838669,
        # 'regressor__estimator__feature_fraction': 0.9881738491942492,
        # 'regressor__estimator__device': 'gpu'
    }

    return CV(model_name='lgbm', n_folds=10, params=params)



def catboost_objective(trial):
    # Define hyperparameters to tune
    params = {
        'regressor__estimator__iterations': trial.suggest_int('regressor__estimator__iterations', 500, 1200),
        'regressor__estimator__learning_rate': trial.suggest_loguniform('regressor__estimator__learning_rate', 0.01, 0.3),
        'regressor__estimator__depth': trial.suggest_int('regressor__estimator__depth', 4, 9),
        'regressor__estimator__l2_leaf_reg': trial.suggest_float('regressor__estimator__l2_leaf_reg', 0.01, 3, log=True),#trial.suggest_float('l2_leaf_reg', 1, 5),
        # 'regressor__estimator__subsample': trial.suggest_float('subsample', 0.5, 0.7),
        # 'regressor__estimator__colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.8, 1),
        'regressor__estimator__random_strength': trial.suggest_float('regressor__estimator__random_strength', 1e-3, 1, log=True),
        'regressor__estimator__bagging_temperature': trial.suggest_float('regressor__estimator__bagging_temperature', 1e-2, 1, log=True)

    }

    return CV(model_name='catboost', n_folds=10, params=params)



In [ ]:
# studies = {}
# for name, model in models.items():
#     study = optuna.create_study(direction='minimize')
#     studies[name] = study

In [ ]:
# objectives = {'xgboost': xgboost_objective, 'catboost': catboost_objective, 'lgbm': lightgbm_objective}
# for name in models:
#     studies[name].optimize(objectives[name], n_trials=15)


In [ ]:
# for name, model in models.items():
#     print(f'{name} Best Parameters:', studies[name].best_params)
#     print(f'{name} Best Log Loss:', studies[name].best_value)

In [ ]:
weights = {'xgboost': 1/3, 'lgbm': 1/3, 'catboost': 1/3}

In [ ]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e1/test.csv')

In [ ]:
ids = df_train['id']
ids

In [ ]:
df_train['year'] = df_train['date'].apply(lambda x: int(x[:4]))
df_train['month'] = df_train['date'].apply(lambda x: int(x[5:7]))
df_train['day'] = df_train['date'].apply(lambda x: int(x[8:10]))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['day of year'] = df_train['date'].dt.dayofyear
df_train['day of week'] = df_train['date'].dt.weekday
df_train['sin day of week'] = np.sin(2 * np.pi * df_train['day of week'] / 7)
df_train['cos day of week'] = np.cos(2 * np.pi * df_train['day of week'] / 7)
df_train['seconds since 1970'] = df_train['date'].astype('int64') // 10 ** 9
df_train.drop('date', axis=1, inplace=True) #
df_train

In [ ]:
# params = {
#     'regressor__estimator__boosting_type': 'gbdt',                     
#     'regressor__estimator__n_estimators': 1249, 
#     'regressor__estimator__learning_rate': 0.04817647633416561, 
#     'regressor__estimator__max_depth': 11, 
#     'regressor__estimator__num_leaves': 260,
#     'regressor__estimator__lambda_l1': 0.0009019590373719743, 
#     'regressor__estimator__lambda_l2': 0.0012757233488032752, 
#     'regressor__estimator__subsample': 0.9051870622183548, 
#     'regressor__estimator__colsample_bytree': 0.8377681417381863, 
#     'regressor__estimator__bagging_freq': 12, 
#     'regressor__estimator__min_data_in_leaf': 83
# }

In [ ]:
# y_pred = np.zeros((df_train.shape[0],))
# for name in models:
#     model = clone(models[name])
#     model.set_params(**studies[name].best_params) # 
#     model.fit(X, y)
#     y_pred += weights[name] * model.predict(df_train)


In [ ]:
# df_pred = pd.DataFrame(y_pred, columns=['num_sold'])
# df_pred.insert(0, 'id', ids)
# df_pred.to_csv('submission.csv', index=False)

In [ ]:
# df_pred